# Step 6/7. Segment Classification & Feature Importance

## 목적
최종 `core_7_KMeans_k5` segment label을 core_7 행동 feature로 재현하는 supervised classification model을 학습하고, segment 구분에 중요한 feature를 확인한다.

## 입력 파일
- `data/analysis/segment_profiling/core_7_KMeans_k5/11_profile_table_with_segment_k5.csv`

## 출력 파일
- `data/analysis/segment_classification/core_7_KMeans_k5/01_classification_metrics_summary.csv`
- model별 classification report와 confusion matrix
- feature importance 및 Logistic Regression coefficient 파일과 plot

## 전체 프로젝트에서의 역할
이 단계는 최종 세그먼트를 신규 고객에게 배정할 수 있는지 확인하는 보조 모델 단계이다. Step 7 feature importance는 segment classification 기준의 중요도를 설명한다.

## 주요 분석 로직
Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM을 비교하고 Accuracy, Macro F1, C3 recall, confusion matrix, feature importance를 확인한다.

## 결과 해석 시 주의점
이 단계는 churn prediction이 아니다. Target은 `Attrition_Flag`가 아니라 최종 K-Means k=5 segment label인 `Cluster`이다. Logistic Regression 성능이 0.99 수준으로 높아도 이는 이탈 예측 정확도가 아니라 segment label reproduction / assignment 성능이다.

### 유틸리티
| 라이브러리 | 용도 |
|---|---|
| `pandas` (pd) | 데이터 로드 및 테이블 처리 |
| `numpy` (np) | 수치 연산 |
| `matplotlib` (plt) | 시각화 |
| `seaborn` | 통계 시각화 |
| `warnings` | 불필요한 경고 메시지 숨김 |

### 모델 및 성능 평가
| 라이브러리 | 용도 |
|---|---|
| `train_test_split` | 학습/테스트 데이터 분리 |
| `LogisticRegression` | 선형 경계 기반 분류 모델 |
| `DecisionTreeClassifier` | 조건 분기 기반 분류 모델 |
| `RandomForestClassifier` | 다수의 트리를 앙상블한 분류 모델 |
| `XGBoost` | 순차적 트리 학습 기반 고성능 모델 |
| `LightGBM` | XGBoost보다 빠른 경량 부스팅 모델 |
| `accuracy_score` / `f1_score` | 정확도 및 F1 점수 계산 |
| `confusion_matrix` | 세그먼트별 예측 혼동 행렬 |
| `classification_report` | 세그먼트별 precision / recall / F1 요약 |

In [ ]:
from pathlib import Path
import sys
import re
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, classification_report,
                              ConfusionMatrixDisplay)
import xgboost as xgb
import lightgbm as lgb

# 저장소 루트를 기준으로 입력과 출력 경로를 고정한다.
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current] + list(current.parents):
        if (path / "data").exists() and (path / "src").exists():
            return path
    return current

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

from src.common import read_csv_safely, configure_korean_font

FINAL_MODEL_KEY = "core_7_KMeans_k5"
DATA_PATH = ROOT / "data" / "analysis" / "segment_profiling" / FINAL_MODEL_KEY / "11_profile_table_with_segment_k5.csv"
OUTPUT_DIR = ROOT / "data" / "analysis" / "segment_classification" / FINAL_MODEL_KEY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

configure_korean_font(verbose=True)

# 파일명에 사용할 안전한 모델 이름을 만든다.
def safe_model_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

print("라이브러리 로드 완료")
print("Project root:", ROOT)
print("Output dir:", OUTPUT_DIR.relative_to(ROOT))

---

# Cell 2 — 데이터 로드 및 구조 확인

Step6/7 분석에 사용할 최종 segment profiling table을 불러온다. 이 파일은 Step 5가 생성한 고객 단위 table이며, `Cluster`가 classification target이다.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Step 6/7 입력 파일이 없습니다: {DATA_PATH.relative_to(ROOT)}\n"
        "먼저 05_segment_profiling.ipynb를 실행하세요."
    )

df = read_csv_safely(DATA_PATH)

print(f"데이터 shape: {df.shape}")
print("\n클러스터 분포:")
print(df["Cluster"].value_counts().sort_index())
print("\n클러스터 레이블:")
print(df.groupby("Cluster")[["Cluster_Label", "Segment_Name"]].first())

-------

# Cell 3 — Feature 선택 및 데이터 분할

## Feature 선택
Classification 입력으로 Step 4 최종 clustering에 사용된 **core_7 행동 스코어**만 사용한다.

아래 컬럼은 feature에서 제외한다.

| 제외 컬럼 | 제외 이유 |
|---|---|
| `Cluster` | Target 변수 |
| `Cluster_Label` | Cluster의 문자열 표현 |
| `Segment_Name` | Cluster 기반 파생 정보 |
| `Attrition_Flag` | 이탈 여부이며 본 단계의 target이 아님 |
| `High_Risk_Signal_Count` | profiling용 사후 검증 변수 |

Target은 최종 K-Means k=5 label인 `Cluster`이다. 이 단계는 이탈 예측이 아니라 segment label assignment를 위한 보조 모델이다.

In [ ]:
# Data Leakage 방지를 위해 core_7 행동 스코어만 입력 feature로 사용한다.
core_7 = [
    "Activity_Score",
    "Count_Decline_Risk_Score",
    "Amount_Decline_Risk_Score",
    "Inactivity_Score",
    "Product_Relationship_Score",
    "Contact_Burden_Score",
    "Credit_Utilization_Score"
]

missing_features = [col for col in core_7 if col not in df.columns]
if missing_features:
    raise ValueError(f"입력 데이터에 core_7 feature가 없습니다: {missing_features}")

X = df[core_7]
y = df["Cluster"].astype(int)
segment_labels = [f"C{i}" for i in sorted(y.unique())]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTest 클래스 분포:\n{y_test.value_counts().sort_index()}")

---

# Cell 4 — 모델 학습 및 성능 비교

Step6의 핵심은 최종 segment label을 얼마나 잘 재현하는지 비교하는 것이다. 여기서 높은 Accuracy와 Macro F1은 churn prediction 성능이 아니라 `core_7_KMeans_k5` label reproduction 성능이다.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree"      : DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random Forest"      : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost"            : xgb.XGBClassifier(n_estimators=200, random_state=42,
                                               eval_metric="mlogloss", verbosity=0),
    "LightGBM"           : lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)
}

results = {}
metric_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    mf1 = f1_score(y_test, y_pred, average="macro")
    results[name] = {"model": model, "pred": y_pred, "accuracy": acc, "macro_f1": mf1}
    metric_rows.append({"model": name, "accuracy": acc, "macro_f1": mf1})

    report_df = pd.DataFrame(
        classification_report(y_test, y_pred, target_names=segment_labels, output_dict=True)
    ).T
    report_df.to_csv(
        OUTPUT_DIR / f"02_classification_report_{safe_model_name(name)}.csv",
        encoding="utf-8-sig"
    )

    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(cm, index=[f"true_{label}" for label in segment_labels], columns=[f"pred_{label}" for label in segment_labels])
    cm_df.to_csv(
        OUTPUT_DIR / f"03_confusion_matrix_{safe_model_name(name)}.csv",
        encoding="utf-8-sig"
    )

    print(f"{name:<25} Accuracy={acc:.4f}  Macro F1={mf1:.4f}")

metrics_df = pd.DataFrame(metric_rows).sort_values("macro_f1", ascending=False)
metrics_df.to_csv(OUTPUT_DIR / "01_classification_metrics_summary.csv", index=False, encoding="utf-8-sig")
display(metrics_df)

## 모델 비교 결과 해석

Step6에서는 <br> **Logistic Regression, <br>
Decision Tree, <br>
Random Forest, <br>
XGBoost, <br>
LightGBM** <br>총 5개의 classification 모델을 비교하였다.  

평가 지표는 **Accuracy**와 **Macro F1-score**를 함께 사용하였다.

### 모델 성능 결과

| 모델 | Accuracy | Macro F1-score |
|---|---:|---:|
| Logistic Regression | 0.9901 | 0.9889 |
| Decision Tree | 0.8934 | 0.8840 |
| Random Forest | 0.9423 | 0.9386 |
| XGBoost | 0.9571 | 0.9544 |
| LightGBM | 0.9580 | 0.9555 |

### 해석

- **Logistic Regression**이 가장 높은 성능을 보였다.
  - Accuracy 0.9901, Macro F1-score 0.9889로 거의 완벽한 수준의 분류 성능을 기록하였다.
  - 이는 core_7 feature가 Step5의 K-Means 세그먼트를 구성하는 핵심 행동 스코어이기 때문에,<br> 세그먼트 경계가 비교적 선형적으로 잘 구분되었기 때문으로 해석할 수 있다.

- **Decision Tree**는 다섯 모델 중 가장 낮은 성능을 보였다.
  - Accuracy 0.8934, Macro F1-score 0.8840으로, 단일 트리 모델의 한계가 드러났다.
  - 이는 세그먼트 경계를 하나의 트리 구조만으로 모두 안정적으로 설명하기 어렵다는 점을 보여준다.

- **Random Forest**는 Decision Tree보다 성능이 크게 향상되었다.
  - Accuracy 0.9423, Macro F1-score 0.9386을 기록하였다.
  - 여러 개의 트리를 앙상블하는 방식으로 안정성이 높아졌으며, 이후 feature importance 해석에도 유용하게 활용할 수 있다.

- **XGBoost**와 **LightGBM**은 모두 높은 성능을 보였다.
  - XGBoost는 Accuracy 0.9571, Macro F1-score 0.9544를 기록하였다.
  - LightGBM은 Accuracy 0.9580, Macro F1-score 0.9555로 XGBoost보다 소폭 높았다.
  - 다만 두 모델 모두 Logistic Regression보다는 낮은 성능을 보였다.

### 종합 해석

이번 Step6에서는 복잡한 비선형 모델보다 **Logistic Regression이 가장 우수한 성능**을 나타냈다.  
이는 본 classification 문제가 일반적인 복잡한 고객 이탈 예측 문제라기보다,<br> 이미 Step5에서 정의된 행동 스코어 기반 세그먼트를 다시 분류하는 문제이기 때문으로 볼 수 있다.  
즉, 세그먼트 경계가 비교적 명확하게 형성되어 있어 단순한 선형 모델만으로도 충분히 높은 성능을 얻을 수 있었다.

따라서 최종 분류 모델로는 **Logistic Regression을 우선적으로 고려할 수 있으며**,  
**Random Forest는 feature importance 해석용 보조 모델**,  
**XGBoost와 LightGBM은 비교 실험용 고성능 모델**로 정리할 수 있다.

---

## 4.1 - Hyperparameter Tuning (선택)

이 셀에서는 Random Forest 모델의 성능을 조금 더 개선할 수 있는지 확인하기 위해  
**GridSearchCV 기반 하이퍼파라미터 튜닝**을 수행한다.

### 목적

기본 Random Forest 모델도 준수한 성능을 보였지만,  
트리 개수나 트리 깊이 같은 하이퍼파라미터 설정에 따라 성능이 달라질 수 있다.

따라서 이 셀에서는  
- `n_estimators`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`

를 후보로 두고,  
3-fold 교차검증을 통해 **Macro F1-score 기준 최적 조합**을 탐색한다.

### 평가 방식

- 탐색 방법: `GridSearchCV`
- 교차검증: `cv=3`
- 평가 지표: `f1_macro`

즉, 전체 세그먼트(C0~C4)를 **균형 있게 잘 분류하는 설정**을 찾는 것이 목적이다.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 8, 12],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(
    rf_base,
    param_grid,
    scoring='f1_macro',  # Macro F1 기준으로 튜닝
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("Best params:", grid_rf.best_params_)
print("Best CV Macro F1:", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_best_rf))
print("Test Macro F1:", f1_score(y_test, y_pred_best_rf, average='macro'))

rf_tuning_result = pd.DataFrame([
    {
        'model': 'Random Forest tuned',
        'best_params': str(grid_rf.best_params_),
        'best_cv_macro_f1': grid_rf.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred_best_rf),
        'test_macro_f1': f1_score(y_test, y_pred_best_rf, average='macro'),
    }
])
rf_tuning_result.to_csv(OUTPUT_DIR / '04_random_forest_tuning_result.csv', index=False, encoding='utf-8-sig')

## Random Forest 하이퍼파라미터 튜닝 결과 해석

Random Forest에 대해 GridSearchCV를 수행한 결과는 다음과 같다.

- 총 탐색 횟수: **24개 후보 조합**
- 총 학습 횟수: **72 fits**  
  (3-fold 교차검증 × 24개 조합)

### 최적 하이퍼파라미터

| 파라미터 | 최적값 |
|---|---|
| `n_estimators` | 200 |
| `max_depth` | None |
| `min_samples_split` | 2 |
| `min_samples_leaf` | 1 |

### 성능 결과

| 지표 | 값 |
|---|---:|
| Best CV Macro F1 | 0.9292 |
| Test Accuracy | 0.9423 |
| Test Macro F1 | 0.9386 |

### 해석

- 최적 하이퍼파라미터는 사실상 **기본 Random Forest 설정과 매우 유사한 형태**로 나타났다.
- 테스트 성능 역시 이전 기본 Random Forest 결과와 거의 동일하였다.
- 즉, 이번 Step6 데이터에서는 Random Forest가 하이퍼파라미터에 아주 민감하게 반응하기보다는,  
  기본 설정만으로도 이미 안정적인 성능을 내고 있었다고 해석할 수 있다.

### 종합 해석

하이퍼파라미터 튜닝을 통해 Random Forest의 성능을 추가로 개선할 가능성을 검토했지만,  
실제 성능 향상은 크지 않았다.

이는 현재 세그먼트 분류 문제가 이미 core_7 행동 스코어를 기반으로 비교적 명확하게 구분되고 있기 때문에,  
복잡한 튜닝보다도 **입력 feature 자체의 구조**가 성능을 결정하는 영향이 더 크다는 점을 시사한다.

따라서 Random Forest는 여전히 해석 보조용 모델로는 유용하지만,  
최종 classification 모델 선택에서는 **Logistic Regression이 더 적절하다**는 기존 결론이 유지된다.

---

## 4.2 - Cross-Validation — 모델 안정성 검증

Train/Test 1회 분할만으로는 결과가 특정 데이터 분할에 의존할 수 있다.  
따라서 **5-Fold Stratified Cross-Validation**을 통해  
각 모델의 성능이 데이터 분할에 관계없이 안정적인지 추가로 검증한다.

- `StratifiedKFold(n_splits=5)` 사용
- 각 fold마다 클래스 비율 유지
- 평가 지표: **Macro F1-score**
- 결과: 5개 fold의 **평균 ± 표준편차**로 표시

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== 5-Fold Stratified Cross-Validation 결과 ===\n")
cv_rows = []
for name, r in results.items():
    model = r["model"]
    cv_scores = cross_val_score(model, X, y,
                                cv=skf,
                                scoring="f1_macro",
                                n_jobs=-1)
    cv_rows.append({"model": name, "cv_macro_f1_mean": cv_scores.mean(), "cv_macro_f1_std": cv_scores.std()})
    print(f"{name:<25} CV Macro F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

cv_summary = pd.DataFrame(cv_rows)
cv_summary.to_csv(OUTPUT_DIR / "05_cross_validation_summary.csv", index=False, encoding="utf-8-sig")

## 5-Fold Stratified Cross-Validation 결과 해석

5-Fold Stratified CV 결과는 다음과 같다.

| 모델 | CV Macro F1 평균 | 표준편차 |
|---|---:|---:|
| Logistic Regression | 0.9921 | ±0.0019 |
| Decision Tree | 0.8818 | ±0.0067 |
| Random Forest | 0.9348 | ±0.0033 |
| XGBoost | 0.9606 | ±0.0059 |
| LightGBM | 0.9583 | ±0.0061 |

### 해석

- **Logistic Regression**은 CV Macro F1 평균이 **0.9921**로 가장 높았다.
  - 표준편차도 **±0.0019**로 다섯 모델 중 가장 낮아, fold에 따른 성능 변동이 거의 없다.
  - 즉, **데이터 분할 방식에 관계없이 가장 안정적이고 일관된 성능**을 보이는 모델임을 확인할 수 있다.

- **Decision Tree**는 평균 0.8818로 가장 낮았고, 표준편차도 ±0.0067로 상대적으로 컸다.
  - 단일 트리 구조의 불안정성이 교차검증 결과에서도 그대로 드러났다.

- **Random Forest**는 평균 0.9348, 표준편차 ±0.0033으로 비교적 안정적이다.
  - 앙상블 효과로 Decision Tree보다 안정성이 크게 향상되었음을 확인할 수 있다.

- **XGBoost**와 **LightGBM**은 각각 0.9606, 0.9583으로 높은 성능을 보였지만,
  - 표준편차가 ±0.0059, ±0.0061로 Logistic Regression보다 변동폭이 컸다.
  - 즉, 부스팅 계열 모델은 성능은 높지만 fold에 따라 소폭 변동이 있었다.

### 종합 해석

Train/Test 1회 분할 결과와 5-Fold CV 결과가 거의 일치하여,  
**이번 Step6 모델 비교 결과가 특정 데이터 분할에 의존하지 않는 안정적인 결과임을 확인**하였다.

특히 Logistic Regression은 평균 성능과 안정성(낮은 표준편차) 모두에서 가장 우수하여,  
최종 모델 선택의 타당성을 교차검증 결과로도 뒷받침할 수 있다.

---

# Cell 5 — 모델 성능 비교 시각화

Cell 4에서 계산한 각 classification 모델의 성능을 **막대 그래프(bar chart)** 로 시각화한다.

### 시각화 목적

- 모델별 **Accuracy**와 **Macro F1-score**를 한눈에 비교
- 단순 정확도뿐 아니라, **세그먼트별 균형 잡힌 분류 성능**까지 함께 확인
- 발표 및 보고서에서 모델 비교 결과를 직관적으로 제시

### 시각화 구성

| 요소 | 설명 |
|---|---|
| x축 | 비교 대상 모델 (Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM) |
| y축 | 성능 점수 |
| 파란 막대 | Accuracy |
| 주황 막대 | Macro F1-score |

### 해석 포인트

- Accuracy와 Macro F1-score가 **모두 높은 모델**일수록 우수한 분류 모델로 해석할 수 있다.
- Accuracy는 높지만 Macro F1-score가 상대적으로 낮다면, 일부 세그먼트에 편향된 예측 가능성을 의심할 수 있다.
- 두 지표가 모두 비슷하게 높다면, 전체 세그먼트를 균형 있게 잘 분류한 것으로 볼 수 있다.

### 추가 설정

- 각 막대 위에 성능 수치를 직접 표시하여 비교를 쉽게 한다.
- 그래프는 `model_comparison.png`로 저장하여 이후 보고서나 발표 자료에 활용할 수 있도록 한다.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import platform

if platform.system() == 'Windows':
    font_path = r"C:\Windows\Fonts\malgun.ttf"  # Windows 한글 폰트
    font_prop = font_manager.FontProperties(fname=font_path)
    plt.rc('font', family=font_prop.get_name())
    plt.rcParams['axes.unicode_minus'] = False  # 마이너스 깨짐 방지

model_names = list(results.keys())
accs = [results[m]['accuracy'] for m in model_names]
mf1s = [results[m]['macro_f1'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, accs, width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, mf1s, width, label='Macro F1', color='coral')

ax.set_ylim(0.8, 1.02)
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15)
ax.set_ylabel('Score')
ax.set_title('Step6 모델 성능 비교 (Accuracy vs Macro F1)')
ax.legend()

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_model_comparison.png', dpi=150)
plt.show()

## 모델 성능 비교 차트 해석

위 차트는 5개 모델의 Accuracy(파란색)와 **Macro F1-score(주황색)** 를 한눈에 비교한 결과다.

### 주요 포인트

- **Logistic Regression**이 Accuracy 0.990, Macro F1 0.989로 **가장 높은 성능**을 기록하였다.
- **Decision Tree**는 Accuracy 0.893, Macro F1 0.884로 **가장 낮은 성능**을 보였다.
- **XGBoost(0.957 / 0.954)**와 **LightGBM(0.958 / 0.956)**은 비슷한 수준의 높은 성능을 보였으나, Logistic Regression보다는 낮았다.
- **Random Forest**는 Accuracy 0.942, Macro F1 0.939로 중간 수준을 유지하였다.

### Accuracy vs Macro F1 차이

모든 모델에서 Accuracy와 Macro F1-score 간 차이가 매우 작다.  
이는 각 모델이 특정 세그먼트에 편향되지 않고 **전체 세그먼트를 고르게 예측**하고 있다는 의미이다.

### 종합 해석

그래프에서 시각적으로도 Logistic Regression이 다른 모델보다 명확히 높은 위치에 있음을 확인할 수 있다.  
단순한 선형 모델임에도 불구하고, core_7 행동 스코어가 이미 세그먼트 경계를 잘 반영하고 있어  
Logistic Regression만으로도 충분히 높은 분류 성능을 달성하였다고 해석할 수 있다.

---


# Cell 6 — Confusion Matrix 비교

이 셀에서는 각 classification 모델의 예측 결과를 **Confusion Matrix(혼동행렬)** 로 시각화한다.

### 분석 목적

- 각 모델이 **어떤 세그먼트를 잘 맞추는지** 확인
- 실제 세그먼트와 예측 세그먼트 사이에서 **어떤 조합이 자주 헷갈리는지** 파악
- 단순 Accuracy 수치만으로는 보이지 않는 **오분류 패턴**을 해석

### Confusion Matrix란?

Confusion Matrix는  
- **행(row)**: 실제 세그먼트  
- **열(column)**: 예측 세그먼트  
를 의미한다.

즉, 대각선 값이 클수록 해당 세그먼트를 정확히 잘 분류한 것이고,  
대각선 밖의 값이 클수록 다른 세그먼트와 혼동한 경우가 많다는 뜻이다.

### 시각화 구성

| 요소 | 설명 |
|---|---|
| 행(row) | 실제 세그먼트 (`y_test`) |
| 열(column) | 예측 세그먼트 (`y_pred`) |
| 대각선 값 | 올바르게 분류된 고객 수 |
| 비대각선 값 | 잘못 분류된 고객 수 |

### 해석 포인트

- 대각선이 진할수록 모델의 분류 성능이 높다고 볼 수 있다.
- 특정 세그먼트에서 비대각선 값이 크다면, 해당 세그먼트가 다른 세그먼트와 행동 패턴이 유사하다는 의미일 수 있다.
- 특히 **초고위험군 C3**가 다른 세그먼트로 자주 분류되는지 확인하는 것이 중요하다.
  - C3를 놓치면 실제 고위험 고객을 제대로 포착하지 못하기 때문이다.

### 활용 방식

이 셀의 결과를 바탕으로  
- 어떤 세그먼트끼리 많이 혼동되는지  
- 어떤 모델이 세그먼트 간 경계를 가장 잘 구분하는지  
를 해석할 수 있다.

또한 이후 Step6 결과 정리에서  
**“C3 recall이 낮은지 / 어떤 세그먼트와 자주 헷갈리는지”** 를 구체적으로 설명하는 근거로 사용할 수 있다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=segment_labels)
    disp.plot(ax=axes[i], colorbar=False)
    axes[i].set_title(name)

axes[-1].axis('off')
plt.suptitle('Confusion Matrix 비교', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_confusion_matrices.png', dpi=150)
plt.show()

## Confusion Matrix 결과 해석

5개 모델의 Confusion Matrix를 비교한 결과, 모델별로 세그먼트 구분 능력에 뚜렷한 차이가 나타났다.

### Logistic Regression

모든 세그먼트에서 대각선 값이 압도적으로 높고, 비대각선(오분류) 값이 거의 없다.

- C0: 471 / 475 정확 분류 → 오분류 4건
- C1: 450 / 452 정확 분류 → 오분류 2건
- C2: 462 / 465 정확 분류 → 오분류 3건
- C3: 430 / 435 정확 분류 → 오분류 5건
- C4: 193 / 199 정확 분류 → 오분류 6건

전체 세그먼트에서 오분류가 거의 발생하지 않아, **가장 이상적인 Confusion Matrix 패턴**을 보였다.

### Decision Tree

5개 모델 중 오분류가 가장 많이 발생하였다.

- C3 오분류가 특히 심각하다.  
  실제 C3 435명 중 356명만 C3로 분류되고, **38명이 C0, 23명이 C1, 12명이 C2로 오분류**되었다.
- C0와 C4에서도 오분류가 다수 발생하였다.
- 단일 트리 구조의 한계로 세그먼트 경계를 안정적으로 학습하지 못한 것으로 해석된다.

### Random Forest

Decision Tree보다 오분류가 크게 줄었지만, Logistic Regression보다는 오분류가 더 많다.

- C3 오분류: 실제 C3 중 394명만 정확 분류, **19명이 C0, 12명이 C1, 8명이 C2로 오분류**
- C0와 C4에서도 소수 오분류 발생
- 앙상블 효과로 Decision Tree보다 안정성이 향상되었음을 확인할 수 있다.

### XGBoost

Random Forest보다 더 향상된 결과를 보였다.

- C3 오분류: 실제 C3 중 410명 정확 분류, **14명이 C0, 3명이 C1, 8명이 C2로 오분류**
- 전반적으로 대각선 값이 뚜렷하지만, Logistic Regression에는 미치지 못하였다.

### LightGBM

XGBoost와 매우 유사한 패턴을 보였으며, 소폭 더 안정적인 결과를 나타냈다.

- C3 오분류: 실제 C3 중 408명 정확 분류, **14명이 C0, 7명이 C1, 6명이 C2로 오분류**
- 전반적으로 XGBoost와 거의 동일한 수준의 성능을 보였다.

### 세그먼트 간 주요 혼동 패턴

모델 간 공통적으로 나타나는 오분류 패턴은 다음과 같다.

| 혼동 패턴 | 발생 모델 | 해석 |
|---|---|---|
| C3 → C0 오분류 | DT, RF, XGB, LGBM | 일부 C3 고객이 C0와 행동 특성이 유사할 수 있음 |
| C3 → C1 오분류 | DT, RF, XGB, LGBM | 일부 C3 고객의 신용 활용 패턴이 C1과 혼동됨 |
| C0 → C3 오분류 | DT에서 두드러짐 | 단일 트리의 세그먼트 경계 불안정성 |

### 종합 해석

Logistic Regression은 모든 세그먼트에서 오분류가 거의 없는 가장 이상적인 결과를 보였다.  
특히 초고위험군 **C3를 다른 세그먼트로 잘못 분류하는 사례가 거의 없어**,  
실무적으로 가장 중요한 고위험 고객을 안정적으로 식별할 수 있음을 확인하였다.

반면 Decision Tree는 C3 오분류가 가장 많아 고위험 고객 탐지 관점에서 가장 부적합한 모델로 나타났다.  
Random Forest, XGBoost, LightGBM은 준수한 성능을 보였지만,  
Logistic Regression의 거의 완벽한 분류 결과와는 차이가 있었다.

---

# Cell 7 — C3(초고위험군) Precision / Recall / F1 비교

이 셀에서는 각 classification 모델이 **초고위험군 세그먼트인 C3를 얼마나 잘 식별하는지**를 별도로 평가한다.

### 분석 목적

전체 Accuracy나 Macro F1-score도 중요하지만,  
실무적으로는 **초고위험군 고객(C3)을 놓치지 않고 잘 잡아내는지**가 더 중요할 수 있다.

특히 카드사 관점에서는 C3 고객을 놓치면  
- 실제 이탈 위험이 높은 고객을 관리 대상에서 제외할 수 있고,
- 유지 전략 적용 시점이 늦어질 수 있기 때문에  
C3에 대한 별도 성능 점검이 필요하다.

### 확인 지표

| 지표 | 의미 |
|---|---|
| `Precision` | 모델이 C3라고 예측한 고객 중 실제로 C3인 비율 |
| `Recall` | 실제 C3 고객 중 모델이 C3로 올바르게 잡아낸 비율 |
| `F1-score` | Precision과 Recall의 균형을 함께 반영한 점수 |

### 왜 Recall이 중요한가?

이번 Step6에서는 특히 **C3 recall**을 중요하게 본다.  
그 이유는 실제 고위험 고객을 놓치지 않고 포착하는 능력이 중요하기 때문이다.

- Recall이 높다 → 실제 C3 고객을 많이 찾아냄
- Recall이 낮다 → 실제 초고위험군을 놓칠 가능성이 큼

따라서 이 셀은  
**“어떤 모델이 고위험 고객을 가장 안정적으로 탐지하는가?”** 를 평가하는 역할을 한다.

In [ ]:
print("=== C3 (초고위험군) Recall 비교 ===\n")
c3_rows = []
for name, r in results.items():
    report = classification_report(y_test, r["pred"],
                                   target_names=segment_labels, output_dict=True)
    c3_recall    = report["C3"]["recall"]
    c3_precision = report["C3"]["precision"]
    c3_f1        = report["C3"]["f1-score"]
    c3_rows.append({"model": name, "c3_precision": c3_precision, "c3_recall": c3_recall, "c3_f1": c3_f1})
    print(f"{name:<25} Precision={c3_precision:.4f}  Recall={c3_recall:.4f}  F1={c3_f1:.4f}")

c3_summary = pd.DataFrame(c3_rows).sort_values("c3_recall", ascending=False)
c3_summary.to_csv(OUTPUT_DIR / "08_c3_performance_summary.csv", index=False, encoding="utf-8-sig")

## C3(초고위험군) 분류 성능 해석

Cell 7에서 각 모델의 C3(초고위험군) Precision, Recall, F1-score를 비교한 결과는 다음과 같다.

| 모델 | Precision | Recall | F1-score |
|---|---:|---:|---:|
| Logistic Regression | 0.9954 | 0.9885 | 0.9919 |
| Decision Tree | 0.8683 | 0.8184 | 0.8426 |
| Random Forest | 0.9314 | 0.9057 | 0.9184 |
| XGBoost | 0.9557 | 0.9425 | 0.9491 |
| LightGBM | 0.9600 | 0.9379 | 0.9488 |

### 결과 해석

- **Logistic Regression**은 C3 분류 성능이 가장 우수했다.
  - Precision 0.9954, Recall 0.9885, F1-score 0.9919로 거의 완벽한 수준이다.
  - 즉, 실제 C3 고객을 거의 놓치지 않으면서도, C3라고 예측한 고객의 정확도도 매우 높았다.

- **Decision Tree**는 다섯 모델 중 가장 낮은 성능을 보였다.
  - Recall이 0.8184로 상대적으로 낮아, 실제 초고위험군 고객을 놓칠 가능성이 가장 크다.
  - 따라서 C3 탐지 관점에서는 최종 모델로 선택하기 어렵다.

- **Random Forest**는 비교적 안정적인 성능을 보였다.
  - Recall 0.9057로 90% 이상을 기록하여 C3를 잘 탐지했지만,
  - Logistic Regression이나 부스팅 계열 모델보다는 낮았다.

- **XGBoost**와 **LightGBM**은 모두 매우 우수한 성능을 보였다.
  - XGBoost는 Recall 0.9425, LightGBM은 Recall 0.9379로 고위험군 탐지 성능이 높았다.
  - 두 모델 모두 Precision과 F1-score도 높게 나타나, C3 식별 능력이 뛰어났다.

### Recall 기준 해석

이번 Step6에서 특히 중요한 지표는 **C3 recall**이다.  
Recall은 실제 초고위험군 고객을 얼마나 놓치지 않고 잘 포착했는지를 의미하기 때문이다.

Recall 기준으로 보면:

1. Logistic Regression — **0.9885**
2. XGBoost — **0.9425**
3. LightGBM — **0.9379**
4. Random Forest — **0.9057**
5. Decision Tree — **0.8184**

즉, **Logistic Regression이 초고위험군 C3를 가장 안정적으로 식별한 모델**이라고 볼 수 있다.

### 종합 해석

전체 성능 비교뿐 아니라,  
초고위험군 C3만 따로 보아도 **Logistic Regression이 가장 우수한 결과**를 보였다.

이는 Step6의 목적이 단순 분류 정확도 확보를 넘어서,  
실제로 중요한 고위험 고객 세그먼트를 안정적으로 예측하는 데 있다는 점에서 매우 의미가 있다.

따라서 C3 탐지 성능까지 고려하면,  
최종 classification 모델로 **Logistic Regression을 선택하는 것이 가장 적절하다**고 해석할 수 있다.
```---



---

# Cell 8 — Random Forest Feature Importance

이 셀에서는 Random Forest 모델을 기준으로,  
각 feature가 고객 세그먼트를 구분하는 데 얼마나 중요한 역할을 하는지 확인한다.

### 분석 목적

Classification 모델이 단순히 세그먼트를 잘 맞추는 것뿐 아니라,  
**어떤 행동 feature가 세그먼트 구분에 핵심적으로 작용했는지 해석**하는 것이 중요하다.

특히 Step6에서는  
- 새로운 고객을 어느 세그먼트에 배정할 수 있는지 예측하는 것과 함께,
- **세그먼트를 구분하는 주요 기준이 무엇인지 설명하는 것**도 중요한 목적이다.

### 왜 Random Forest를 사용하는가?

Random Forest는 여러 개의 Decision Tree를 기반으로 동작하는 앙상블 모델이며,  
학습 과정에서 각 feature가 분류 성능 향상에 얼마나 기여했는지를 계산할 수 있다.

따라서 Random Forest의 feature importance는  
**세그먼트 분류 기준을 직관적으로 해석하는 도구**로 활용할 수 있다.

### 시각화 구성

| 요소 | 설명 |
|---|---|
| y축 | core_7 행동 feature |
| x축 | Feature Importance 값 |
| 막대 길이 | 해당 feature가 세그먼트 분류에 기여한 상대적 중요도 |

### 해석 포인트

- importance 값이 클수록 세그먼트 구분에 더 큰 영향을 준 feature로 볼 수 있다.
- importance 값이 매우 낮은 feature는 상대적으로 구분 기여도가 작다고 해석할 수 있다.
- 이를 통해 **카드 고객 세그먼트가 어떤 행동 신호를 중심으로 나뉘는지** 설명할 수 있다.

In [ ]:
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=core_7).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importance')
ax.set_xlabel('Importance')

for i, v in enumerate(importances):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
importances.sort_values(ascending=False).rename('importance').to_csv(OUTPUT_DIR / '09_feature_importance_random_forest.csv', encoding='utf-8-sig')
plt.savefig(OUTPUT_DIR / '10_feature_importance_random_forest.png', dpi=150)
plt.show()

## Random Forest Feature Importance 결과 해석

Random Forest 기준 feature importance를 확인한 결과,  
세그먼트 구분에 중요한 행동 feature는 다음과 같이 나타났다.

| 순위 | Feature | Importance |
|---|---|---:|
| 1 | `Credit_Utilization_Score` | 0.266 |
| 2 | `Activity_Score` | 0.203 |
| 3 | `Product_Relationship_Score` | 0.182 |
| 4 | `Count_Decline_Risk_Score` | 0.154 |
| 5 | `Amount_Decline_Risk_Score` | 0.129 |
| 6 | `Contact_Burden_Score` | 0.049 |
| 7 | `Inactivity_Score` | 0.017 |

### 주요 해석

- 가장 중요한 feature는 **`Credit_Utilization_Score`** 로 나타났다.
  - 이는 **고객이 신용한도를 얼마나 활용**하고 있는지가 세그먼트 구분에 가장 큰 영향을 주었다는 의미이다.
  - 즉, 카드 의존도나 한도 활용 패턴이 고객군을 나누는 핵심 기준 중 하나로 작용했음을 시사한다.

- 두 번째로 중요한 feature는 **`Activity_Score`** 였다.
  - 고객의 **거래 빈도, 거래 금액, 전반적인 카드 사용 활성도**가 세그먼트 구분에 큰 역할을 했다고 볼 수 있다.
  - 이는 활동성이 고객 세그먼트의 기본 축이라는 점을 보여준다.

- 세 번째는 **`Product_Relationship_Score`** 로 확인되었다.
  - 고객이 카드사와 얼마나 **다양한 관계 상품을 보유**하고 있는지가 세그먼트 분류에 중요한 정보로 작용하였다.
  - 즉, 관계 강도가 고객군 구분의 주요 기준 중 하나였음을 의미한다.

- **`Count_Decline_Risk_Score`** 와 **`Amount_Decline_Risk_Score`** 도 비교적 높은 중요도를 보였다.
  - 이는 **최근 거래 횟수 감소**와 **거래 금액 감소**가 세그먼트 구분에 의미 있는 신호라는 점을 보여준다.
  - 특히 거래 감소 관련 변수들이 함께 중요하게 나타난 것은, 고객 충성도 약화나 이탈 위험 신호가 세그먼트 구조와 연결되어 있음을 시사한다.

- 반면 **`Contact_Burden_Score`** 와 **`Inactivity_Score`** 는 상대적으로 낮은 중요도를 보였다.
  - **고객 접촉 부담**과 **비활성 기간**은 일정 부분 의미가 있지만,
  - 이번 k=5 세그먼트 구조에서는 다른 feature들에 비해 구분 기여도가 상대적으로 작았다고 해석할 수 있다.

### 종합 해석

전체적으로 보면,  
이번 세그먼트는 **한도 활용도 > 카드 사용 활성도 > 관계 강도 > 거래 감소 신호**를 중심으로 구분된다고 해석할 수 있다.

즉, 고객 세그먼트는 단순히 “최근 비활성 고객인가”만으로 나뉘는 것이 아니라,  
- 카드를 얼마나 적극적으로 쓰는지,
- 최근 사용이 감소하고 있는지,
- 카드사와의 관계가 얼마나 깊은지,
- 신용한도를 어떤 방식으로 활용하는지  
와 같은 복합적 행동 패턴에 의해 형성된 것으로 볼 수 있다.

따라서 Step6의 feature importance 결과는  
Step5에서 정의된 세그먼트가 **행동 기반 고객 구조를 반영하고 있다**는 점을 뒷받침하는 해석 근거로 활용할 수 있다.
```---


---

## 8.1 - Logistic Regression Coefficient 분석

Random Forest의 feature importance가  
"전체적으로 어떤 feature가 세그먼트 구분에 중요한가"를 보여준다면,  
**Logistic Regression의 coefficient**는  
"각 feature가 특정 세그먼트를 예측하는 데 어떤 방향으로 작용하는가"를 보여준다.

### 해석 방법

| 계수 방향 | 의미 |
|---|---|
| 양수(+) | 해당 feature가 클수록 그 세그먼트일 가능성이 높아짐 |
| 음수(-) | 해당 feature가 클수록 그 세그먼트일 가능성이 낮아짐 |
| 절댓값이 클수록 | 해당 feature의 영향력이 더 강함 |

히트맵에서 **빨간색**은 양의 계수(강하게 영향),  
**파란색**은 음의 계수(반대 방향 영향)를 의미한다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

lr_model = results['Logistic Regression']['model']
# 계수 DataFrame 생성
coef_df = pd.DataFrame(
    lr_model.coef_,
    index=segment_labels,
    columns=core_7
)

print("=== Logistic Regression Coefficient ===")
print(coef_df.round(4))
coef_df.to_csv(OUTPUT_DIR / '11_logistic_regression_coefficients.csv', encoding='utf-8-sig')
coef_df.to_csv(OUTPUT_DIR / '11_logistic_regression_coefficients.csv', encoding='utf-8-sig')

# 히트맵 시각화
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(coef_df.values, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(core_7)))
ax.set_xticklabels(core_7, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(len(segment_labels)))
ax.set_yticklabels(segment_labels)
ax.set_title('Logistic Regression Coefficient Heatmap')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '12_logistic_regression_coefficient_heatmap.png', dpi=150)
plt.show()

## Logistic Regression Coefficient 해석

Logistic Regression의 각 세그먼트별 계수(coefficient)는 다음과 같다.

| Feature | C0 | C1 | C2 | C3 | C4 |
|---|---:|---:|---:|---:|---:|
| Activity_Score | +0.78 | +0.52 | **+7.13** | **-5.16** | -3.27 |
| Count_Decline_Risk_Score | -0.05 | +0.24 | -0.08 | **+6.84** | **-6.96** |
| Amount_Decline_Risk_Score | +1.06 | +1.59 | +0.91 | **+4.75** | **-8.31** |
| Inactivity_Score | -0.61 | +0.34 | -0.44 | +1.52 | -0.82 |
| Product_Relationship_Score | **+5.02** | +1.68 | **-7.15** | -1.10 | +1.55 |
| Contact_Burden_Score | -1.33 | -0.86 | -2.03 | **+3.98** | +0.23 |
| Credit_Utilization_Score | -3.82 | **+8.90** | -2.17 | -2.97 | +0.06 |

양수(+)는 해당 feature가 클수록 그 세그먼트일 확률이 높아진다는 의미이고,  
음수(-)는 그 반대를 의미한다. 절댓값이 클수록 영향력이 강하다.

### 세그먼트별 주요 해석

- **C0 — 상품 관계 중심 저위험 안정군** 
  - `Product_Relationship_Score` 계수가 **+5.02**로 가장 높다.
  - 관계 상품 수가 많을수록 C0로 분류될 가능성이 높음을 의미한다.
  - 즉, C0는 카드사와의 관계가 깊은 안정적 고객군임을 계수가 뒷받침한다.

- **C1 — 신용 활용 중심 저위험 금융활용군** 
  - `Credit_Utilization_Score` 계수가 **+8.90**으로 압도적으로 높다.
  - 신용한도를 적극 활용하는 고객일수록 C1로 분류된다는 의미이다.
  - C1 세그먼트가 "신용 활용 중심 금융활용군"으로 명명된 것과 정확히 일치한다.

- **C2 — 고활동·저관계 잠재 전환군** 
  - `Activity_Score` 계수가 **+7.13**으로 매우 높고,
    `Product_Relationship_Score`는 **-7.15**로 강한 음의 값을 보인다.
  - 즉, 카드를 활발하게 쓰지만 관계 상품 수는 적은 고객이 C2로 분류된다.
  - C2 설명("활동성은 높지만 상품 관계가 약한 잠재 전환 고객군")과 완벽하게 일치한다.

- **C3 — 거래 감소·저활동·접촉 부담형 초고위험 이탈군** 
  - `Count_Decline_Risk_Score` **+6.84**, `Amount_Decline_Risk_Score` **+4.75**,
    `Contact_Burden_Score` **+3.98**으로 모두 강한 양의 계수를 보인다.
  - 반면 `Activity_Score`는 **-5.16**으로 강한 음의 값이다.
  - 즉, 거래 횟수·금액이 감소하고, 고객 접촉이 잦으면서, 활동성이 낮을수록 C3로 분류된다.
  - 이는 초고위험 이탈군의 핵심 행동 신호를 계수가 직접 보여주는 결과다.

- **C4 — 저활동·비감소형 초저위험 안정군** 
  - `Count_Decline_Risk_Score` **-6.96**, `Amount_Decline_Risk_Score` **-8.31**로
    거래 감소 신호가 낮을수록 C4로 분류된다.
  - C4는 활동성은 낮지만 거래 감소 신호가 없는 안정적 고객군으로,
    C3와 반대되는 계수 패턴을 보인다.

### 종합 해석

Logistic Regression의 coefficient는 Random Forest의 feature importance와 달리 <br> **방향성(+/-)까지 해석할 수 있다**는 점에서 추가적인 인사이트를 제공한다.

특히 각 세그먼트를 구분하는 핵심 feature가 계수 크기로 명확히 드러났으며,  
Step5에서 정의한 세그먼트 특성 설명과 일치하는 방향으로 계수가 형성되어  
**분류 모델이 세그먼트 구조를 잘 학습하고 있음을 해석적으로도 확인**할 수 있다.

---

# Cell 9 — 최종 선택 모델 세그먼트별 분류 성능

이 셀에서는 Step6에서 최종 모델로 선택한 **Logistic Regression**의  
세그먼트별 분류 성능을 상세하게 확인한다.

### 분석 목적

앞선 셀에서는 여러 classification 모델의 전체 성능(Accuracy, Macro F1-score)을 비교하였다.  
하지만 최종 모델을 선택한 뒤에는,  
각 세그먼트(C0~C4)에 대해 **얼마나 정확하게 분류했는지 세부적으로 확인할 필요**가 있다.

따라서 이 셀에서는 `classification_report`를 사용하여  
세그먼트별 **Precision, Recall, F1-score, Support**를 출력한다.

### 지표 설명

| 지표 | 의미 |
|---|---|
| `Precision` | 해당 세그먼트라고 예측한 고객 중 실제로 맞는 비율 |
| `Recall` | 실제 해당 세그먼트 고객 중 올바르게 찾아낸 비율 |
| `F1-score` | Precision과 Recall을 함께 반영한 조화 평균 |
| `Support` | 해당 세그먼트의 실제 고객 수 |

### 활용 목적

이 셀의 결과를 통해  
- 특정 세그먼트에서 성능이 낮은 부분이 있는지 확인하고,
- 최종 모델이 전체적으로 균형 잡힌 예측을 수행하는지 검토할 수 있다.

특히 C3처럼 중요한 고위험 세그먼트뿐 아니라,  
다른 세그먼트(C0~C4)까지 고르게 잘 맞추는지를 확인하는 데 의미가 있다.

In [ ]:
best_model_name = "Logistic Regression"
best_pred = results[best_model_name]["pred"]

print(f"=== {best_model_name} 세그먼트별 분류 성능 ===")
best_report = classification_report(y_test, best_pred, target_names=segment_labels)
print(best_report)

best_report_df = pd.DataFrame(
    classification_report(y_test, best_pred, target_names=segment_labels, output_dict=True)
).T
best_report_df.to_csv(OUTPUT_DIR / "13_final_logistic_regression_classification_report.csv", encoding="utf-8-sig")

summary_text = f"""# Step 6/7 Segment Classification 요약

## 목적
최종 `core_7_KMeans_k5` segment label을 core_7 행동 feature로 재현하는 classification model을 비교하였다.

## 해석 주의
이 결과는 churn prediction 성능이 아니다. Target은 `Attrition_Flag`가 아니라 최종 K-Means k=5 label인 `Cluster`이다. 높은 Logistic Regression 성능은 segment label reproduction / assignment 성능으로 해석한다.

## 주요 출력
- `01_classification_metrics_summary.csv`
- `07_confusion_matrices.png`
- `09_feature_importance_random_forest.csv`
- `11_logistic_regression_coefficients.csv`
- `13_final_logistic_regression_classification_report.csv`
"""
(OUTPUT_DIR / "14_step6_7_classification_summary.md").write_text(summary_text, encoding="utf-8")
print("출력 저장 위치:", OUTPUT_DIR.relative_to(ROOT))

## Logistic Regression 세그먼트별 분류 성능 해석

최종 선택 모델인 **Logistic Regression**의 세그먼트별 분류 성능은 다음과 같다.

| Segment | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| C0 | 0.99 | 0.99 | 0.99 | 475 |
| C1 | 0.99 | 1.00 | 0.99 | 452 |
| C2 | 0.99 | 0.99 | 0.99 | 465 |
| C3 | 1.00 | 0.99 | 0.99 | 435 |
| C4 | 0.99 | 0.97 | 0.98 | 199 |

### 전체 성능 요약

- 전체 Accuracy는 **0.99**
- Macro average Precision / Recall / F1-score도 모두 **0.99**
- Weighted average 역시 모두 **0.99**

즉, Logistic Regression은 전체적으로 매우 높은 수준의 세그먼트 분류 성능을 보였다.

### 세그먼트별 해석

- **C0**
  - Precision, Recall, F1-score가 모두 0.99로 매우 우수하다.
  - C0 세그먼트는 거의 완벽하게 구분된다고 볼 수 있다.

- **C1**
  - Recall이 1.00으로 나타났다.
  - 이는 실제 C1 고객을 거의 모두 정확히 식별했다는 의미이다.
  - Precision과 F1-score도 0.99로 매우 안정적이다.

- **C2**
  - Precision, Recall, F1-score가 모두 0.99로 매우 높다.
  - C2 역시 다른 세그먼트와 거의 혼동되지 않고 잘 구분되었다.

- **C3**
  - Precision 1.00, Recall 0.99, F1-score 0.99를 기록하였다.
  - 특히 Precision이 1.00이라는 점은, 모델이 C3라고 예측한 고객이 거의 모두 실제 C3였다는 뜻이다.
  - Recall도 0.99로 높아, 초고위험군 고객을 거의 놓치지 않고 식별한 것으로 해석할 수 있다.

- **C4**
  - Precision 0.99, Recall 0.97, F1-score 0.98로 다른 세그먼트보다 소폭 낮지만 여전히 매우 높은 수준이다.
  - 즉, C4가 상대적으로 약간 더 구분이 어려운 세그먼트일 수 있으나, 전체적으로는 충분히 우수한 성능을 보였다.

### 종합 해석

최종적으로 Logistic Regression은  
C0~C4 모든 세그먼트에서 **매우 균형 잡힌 분류 성능**을 보였다.

특히 이번 Step6에서 중요하게 본 **초고위험군 C3**에 대해서도  
Precision과 Recall 모두 거의 완벽한 수준을 기록하여,  
고위험 고객을 놓치지 않으면서도 정확하게 식별할 수 있음을 보여주었다.

또한 소수 세그먼트인 C4에서도 높은 성능을 유지하고 있어,  
특정 세그먼트에 편향되지 않고 전체 고객군을 안정적으로 분류하는 모델이라고 해석할 수 있다.

따라서 Step6의 최종 classification 모델로  
**Logistic Regression을 선택하는 것이 가장 적절하다**고 정리할 수 있다.

---

## Step6 요약

Step6에서는 Step5에서 확정한 **K-Means k=5 고객 세그먼트(Cluster)** 를 target으로 두고,  
core_7 행동 feature를 기반으로 새로운 고객이 어느 세그먼트에 속하는지 예측하는 classification 모델을 구축하였다.

### 분석 목적

이번 Step6의 목적은 단순한 이탈 여부 예측이 아니라,  
고객의 행동 패턴을 바탕으로 **어느 세그먼트에 속하는지 예측**하여  
향후 세그먼트별 유지 전략에 활용할 수 있는 분류 모델을 만드는 것이었다.

### 사용 모델

비교한 모델은 다음과 같다.

- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- LightGBM

또한 Random Forest에 대해서는 GridSearchCV 기반의 **하이퍼파라미터 튜닝**도 추가로 수행하였다.

### 주요 결과

모델 비교 결과, **Logistic Regression**이 가장 우수한 성능을 보였다.

| 모델 | Accuracy | Macro F1-score |
|---|---:|---:|
| Logistic Regression | 0.9901 | 0.9889 |
| Decision Tree | 0.8934 | 0.8840 |
| Random Forest | 0.9423 | 0.9386 |
| XGBoost | 0.9571 | 0.9544 |
| LightGBM | 0.9580 | 0.9555 |

특히 초고위험군인 **C3 세그먼트**에 대해서도 Logistic Regression이 가장 높은 성능을 보였다.

| 모델 | C3 Precision | C3 Recall | C3 F1-score |
|---|---:|---:|---:|
| Logistic Regression | 0.9954 | 0.9885 | 0.9919 |
| Decision Tree | 0.8683 | 0.8184 | 0.8426 |
| Random Forest | 0.9314 | 0.9057 | 0.9184 |
| XGBoost | 0.9557 | 0.9425 | 0.9491 |
| LightGBM | 0.9600 | 0.9379 | 0.9488 |

### 추가 해석

Confusion Matrix를 통해 세그먼트 간 혼동 패턴을 확인한 결과,  
Logistic Regression은 대부분의 세그먼트를 거의 완벽하게 구분하였다.  
반면 Decision Tree는 세그먼트 간 오분류가 상대적으로 많았고,  
Random Forest, XGBoost, LightGBM은 우수한 성능을 보였지만 Logistic Regression에는 미치지 못했다.

Random Forest의 feature importance를 확인한 결과,  
세그먼트 구분에 중요한 변수는 다음과 같았다.

1. `Credit_Utilization_Score`
2. `Activity_Score`
3. `Product_Relationship_Score`
4. `Count_Decline_Risk_Score`
5. `Amount_Decline_Risk_Score`

즉, 고객 세그먼트는  
**신용한도 활용도, 카드 사용 활성도, 관계 강도, 거래 감소 신호**를 중심으로 구분되고 있음을 확인할 수 있었다.

또한 Random Forest 하이퍼파라미터 튜닝을 수행했지만,  
기본 모델 대비 성능 향상은 크지 않았다.  
이는 이번 세그먼트 분류 문제에서 복잡한 튜닝보다도,  
core_7 행동 feature 자체가 이미 강한 설명력을 가지고 있음을 시사한다.

### 최종 결론

Step6의 최종 classification 모델로는 **Logistic Regression**을 선택하는 것이 가장 적절하다.

그 이유는 다음과 같다.

- Accuracy와 Macro F1-score가 가장 높음
- 초고위험군 C3에 대한 Recall이 가장 높음
- 세그먼트별 분류 성능이 전반적으로 매우 균형적임
- 모델이 단순하고 해석 가능성이 높음
- 새로운 고객을 빠르게 세그먼트에 배정하는 데 적합함

따라서 Step6 결과는  
Step5에서 정의된 고객 세그먼트를 **실제로 예측 가능한 분류 체계로 연결했다는 점**에서 의미가 있으며,  
이후 Step7의 feature importance 비교 및 최종 보고서 작성의 기반으로 활용할 수 있다.